<h1 style="color: #008080;"> Using PaddleOCR on The Spiritualist Newspaper (1869 - 1882)</h1>


<p align="center">
  <img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQv4j6OMs2Ljhx28v8Dv9fhqEp10Wk29j9QsQ&s" width="500" alt="PaddleOCR Logo">
</p>

This notebook was created in November 2025 for the National Library of Scotland (NLS) by Dr Joe Nockels, Digital Fellow. His related project, 'Recognising Text, Recognising Processes: eXplainable Automatic Text Recognition for Scottish Spiritualist Newspapers', was made possible through support from the NLS and Digital Humanities Institute (DHI), University of Sheffield. Images used of *The Spiritualist Newspaper* were first captured by the NLS Digitisation Unit in 2019, and made accessible as an open resource via the NLS Data Foundry (https://data.nls.uk/data/digitised-collections/spiritualist-newspapers/). In extracting text from these images, this project conducted technical trials of available OCR engines representative of current developmental and business structures, from open source applications to commercial LLMs. This also enabled improvements to be made on the NLS's inaccurate legacy OCR for *The Spiritualist*.

This research aimed to establish how far Automatic Text Recognition can coalesce with 'eXplainable' developmental principles, and the extent to which such principles can aid non-technical library users in their understanding of how AI reaches its transcription results. As such, the project's OCR experiments occurred alongside a thematic coding of each tool's public-facing documentation, including how-to guides, model cards and GitHub README files, as well as other grey literature such as technical papers produced by developmental consortia. This was done through a bespoke web crawler using in-built request delays to ensure server compliance, with eXplainable themes coded via NVIVO (https://lumivero.com/products/nvivo/).

The resultant OCR transcription data created through this experimentation is hosted via a comparative web dashboard developed by the DHI, linked to as part of the Jupyter Notebook seires. This platform will enable visual and curatorial comparisons between each AI-enabled tool's transcription results, alongside interactive hosting of *The Spiritualist* images used. The site also provides a clear link to each OCR engine notebook. Together, the project team hope that such a resource will enable curators, library users and Digital Humanists to move beyond a purely single-score quantitative benchmarking of OCR tools, such as Character and Word Error Rate (CER/WER), toward a more curatorial assessment of AI-enabled transcription outputs, especially considering that AI-enabled transcription is now being applied to qualitative domains such as text interpretation and summarisation.

As further context, this Jupyter Notebook follows previous NLS projects using the format to provide introductory analyses to Data Foundry collections, in line with the OpenGLAM movement and Collections as Data principles (Ames & Havens, 2020). As with these projects, this work was conducted to encourage, enable and support computational research methods on collections, as well as meeting FAIR principles for digital research (https://www.go-fair.org/fair-principles/). The need for human-readable explanations for digital cultural heritage research, with Jupyter Notebooks being a prime example, was also expressed during the 2025 Bridging Responsible AI Divides Cultural Heritage Forum at Senate House Library, where initial findings from this OCR work were first presented.

In following FAIR principles, please contact Joe Nockels at j.nockels@sheffield.ac.uk if any of the following code requires clarification, troubleshooting, or - indeed - you would like to make use of the web crawler used on OCR documentation, entitled 'GhostCrawler'. 

<h4> About The Spiritualist Newspaper </h4>

*The Spiritualist Newspaper*,first published by E.W. Allen (London) in 1869, forms a key source for how those interested in physical investigations and spiritual forces communicated with the deceased, argued about issues of the day, as well as publicised gatherings. The Newspaper followed a wake of spiritualist activity in Britain, directed by earlier 1850s American Transcendentalism and a more genral reaction to scientific naturalism. 

*The Spiritualist* was chosen as an OCR case study for several reasons: 

- Scottish spiritualist histories remain underrepresented, as Foot (2023) discusses. In our case, this partially results from historical detail being contained in heterogeneous and hard-to-transcribe material. This makes accurate OCR recognition more challenging, especially without requisite training data. With a fully searchable sample of 50 pages of *The Spiritualist* enabled through our OCR testing and reference, manually corrected, transcription, spiritualist activity outside of Scotland's Central Belt can be further explored. The trained reference model will also be applied to the entire collection as future work.

- In addition, Foot (2023: 73) suggests that seance mediums underwent 'automatic writing' to channel spiritual forces. The latter insight forms a neat parallel to charting the extent of unexplainable OCR processes in the current technical offering to libraries, as well as the potentially opaque behaviour and outputs of AI-enabled transcription models.

Alongside these two primary reasons, the project also aimed to provide metrics for the standardisation of OCR transcription at the NLS, with the Library currently undergoing a reassessment of its Foxit / ABBYY FineReader license. The timings registered through the below trials, alongside the CER of outputs, were used as an evidence base for NLS decision-making around digital collection transcription.

 -  **Data format**: png. manuscript images, pre-processed (thresholding and additional cropping) in R Studio using Image Magick. 
 -  **Data creation process**: Vision-Language Model (VLM) Optical Character Recognition.
 - **Data source**: https://data.nls.uk/data/digitised-collections/spiritualist-newspapers/, 50 pages of the first 1869 edition.

<h4> Table of Contents </h4>

- <a href = '#section 0'><h6>About the Model</h6></a>
- <a href = '#section 1'><h6>Preparation</h6></a>
- <a href = '#section 2'><h6>OCR Processing</h6></a>
- <a href = '#section 3'><h6>Post-Processing</h6></a>

<a href = '#section 0'><h4>About the Model</h4></a>

The following notebook deploys PaddleOCR (https://doi.org/10.48550/arXiv.2510.14528), a vision-language model (VLM) that integrates NaViT-style visual encoding into text extraction processes. In this case, Paddle’s Standard English OCR was used, with its paddlex layout pre-processing removed due to conflicts with Mac IOS, caused by the researcher using an Apple Mac Silicion. Instead, OpenCV Graphics (https://opencv.org) - a standard Python package for column detection and layout analysis - was used, as with our Tesseract and EasyOCR experiments. The following code also includes several other tweaks to ensure effective running, model importation and Mac IOS compliance.

<a href = '#section 1'><h4>Preparation</h4></a>

With our tests being ran on an Apple Mac Silicon (M1), the following code uses an environment compatible with Arm64 instead of the standard x86_64 Intel / Anaconda set-up. This can be checked within Jupyter Notebook, through the following code: 

import platform
platform.machine()
print(platform.machine())


Arm64 relies on miniforge for PaddleOCR to run effectively - first, download the .sh file from: https://github.com/conda-forge/miniforge/releases/latest. Due to .extension incompatibility between Mac IOS and PaddleOCR, paddlex must also be removed here and uninstalled. This is a legacy dependency for newer PaddleOCR versions and not needed to run the model efficiency. However, even the latest PaddleOCR models will revert to calling it, which causes typing_extension conflicts around Sentinel as well as the .extension issue. You will therefore need to ensure that paddlex is not present at all across the model build.

Also, to deploy the following code on a similar environment to our Apple Mac Silicon, ensure you have the newer 2.7 version of PaddleOCR, and version 2.6.1 of the Paddle Paddle base. With the build being sensitive, it is best to pre-process your images elsewhere, instead of trying to wrangle paddlex in a separate environment. We advise using Image Magick, as we have done to threshold and perform any additional cropping needed, either through Python or R Studio.

After it is clear paddlex has been removed, make sure your conda environment paddle_env is activated within your bash terminal - conda activate paddle_env, and the kernel is registered in Jupyter Notebook as Arm64. For more advice on using conda, see: https://anaconda.org/anaconda/conda

**Step 1** - First of all, let us construct an exception to confirm that paddlex is not installed.

In [ ]:
# Check if paddlex is installed
try:
    import paddlex
    print("paddlex IS installed! Version:", paddlex.__version__)
except ModuleNotFoundError:
    print("paddlex is NOT installed.")

In [ ]:
!pip list | grep paddle

In [ ]:
!pip uninstall -y paddlex

**Step 2** - After confirming that the paddlex package is removed, we can import PaddleOCR and test it out-of-the-box on the first *Spiritualist* sample page. This also serves as a test for the base environment, since the PaddleOCR build is particularly sensitive to package versions and installations.

In [ ]:
# first page OCR test 

from paddleocr import PaddleOCR # make sure than numpy is downgraded to 1.23, due to 'int' support, and that polygon and lanms are installed 

ocr = PaddleOCR(lang='en') # deploy PaddleOCR engine
result = ocr.ocr("/Users/joenockels/Spiritualist_Sample/0001.png") # replace with your file path
print(result)

**Step 3** - As further verification that PaddleOCR is working as intended, the following cell deploys OpenCV Graphics across the layout of our first page test. This provides a visual, curatorial, verification for whether OpenCV can suitably replace paddlex in our OCR workflow, as well as if and where inaccuracies occur and their impact on PaddleOCR text recognition.

In [ ]:
import numpy as np # for calculation of word coordinates and error rates
import cv2 # for layout analysis

img_path = "/Users/joenockels/Spiritualist_Sample/0001.png" # again, replace with your own directory and file path
result = ocr.ocr(img_path)

image = cv2.imread(img_path)

for line in result: # detect boundary boxes in the page layout
    for box, (text, score) in line:
        box = [tuple(map(int, pt)) for pt in box]
        cv2.polylines(image, [np.array(box)], isClosed=True, color=(0,255,0), thickness=2)

cv2.imwrite("page1_boxes.png", image)

**Step 4** - The final part of our preparation stage is to import the required packages for measuring OCR processing time, a useful metric for NLS discussions around OCR licence changes at-scale, as well as constructing a file structure whereby images can be easily uploaded and .txt and ALTO XML (https://www.loc.gov/standards/alto/.outputs) exported. The former file format is needed to conduct external CER calculations via the local host application CERberus (https://github.com/WHaverals/CERberus), and the latter for the interactive hosting of *Spiritualist* images via the DHI's web dashboard.

In [ ]:
import os
import time
from paddleocr import PaddleOCR

ocr = PaddleOCR(lang='en')

input_folder = "/Users/joenockels/Spiritualist_Sample" # replace with your own directory and file path
output_file = "/Users/joenockels/Spiritualist_Sample.txt"

# Get all image files sorted
image_files = sorted([
    os.path.join(input_folder, f)
    for f in os.listdir(input_folder)
    if f.lower().endswith((".png", ".jpg", ".jpeg", ".tif"))
])

start_time = time.time()

all_results = []

In [ ]:
<a href = '#section 2'><h6>OCR Processing</h6></a>

<a href = '#section 2'><h4>OCR Processing</h4></a>

The following code runs PaddleOCR over the full 50 pg. *Spiritualist* sample through a coded loop process. We also provide the code for a Jupyter widget to print each page's processing confirmation, so the user can track PaddleOCR's progress. Page transcriptions are then appended together into one continous .txt file, with optional separators between pages. In our case, these separators were commented out - due to their inclusion within the .txt file leading to insertion errors when calculating CER within CERberus.

This code also provides a printed total OCR processing time, as well as a page breakdown. 

In [ ]:
# OCR and write to file in one pass
with open(output_file, "w", encoding="utf-8") as out_txt:
    for img_path in image_files:
        print(f"Processing {img_path}...")
        result = ocr.ocr(img_path)
        all_results.append(result)

        # Extract recognised text line by line
        for line in result:
            text = line[1][0] 
            out_txt.write(text + "\n")

        out_txt.write("\n" + "-"*50 + "\n\n")  # optional separator between pages

end_time = time.time()
total_time = end_time - start_time
print(f" Processed {len(image_files)} pages in {total_time:.2f} seconds")
print(f"Text saved to: {output_file}")

<a href = '#section 3'><h4>Post-Processing</h4></a>

With the OCR .txt generated using PaddleOCR, an ALTO XML generation function is needed to enable the interactive hosting of *Spiritualist* images via the DHI. This cell block provides an ALTO XML compliant scaffold, which aims to produce IIIF (https://iiif.io) compliant XML and interactive OCR transcriptions for users' curatorial comparison. 

In [ ]:
import cv2
import os
import xml.etree.ElementTree as ET # for XMl construction
from paddleocr import PaddleOCR

ocr = PaddleOCR(lang='en')
input_folder = "/Users/joenockels/Spiritualist_Sample" # replace with your own directory and file path
output_folder = "/Users/joenockels/Spiritualist_Output"

os.makedirs(output_folder, exist_ok=True)

def create_alto_xml(ocr_results, img_width, img_height):
    root = ET.Element('alto', xmlns="http://www.loc.gov/standards/alto/ns-v4#")
    layout = ET.SubElement(root, 'Layout')
    page = ET.SubElement(layout, 'Page', WIDTH=str(img_width), HEIGHT=str(img_height))
    print_space = ET.SubElement(page, 'PrintSpace', HEIGHT=str(img_height), WIDTH=str(img_width))

    for line in ocr_results:
        for box, (text, score) in line:
            x_min = int(min([pt[0] for pt in box]))
            y_min = int(min([pt[1] for pt in box]))
            x_max = int(max([pt[0] for pt in box]))
            y_max = int(max([pt[1] for pt in box]))

            text_block = ET.SubElement(print_space, 'TextBlock',
                                       HPOS=str(x_min), VPOS=str(y_min),
                                       WIDTH=str(x_max-x_min), HEIGHT=str(y_max-y_min))
            text_line = ET.SubElement(text_block, 'TextLine',
                                      HPOS=str(x_min), VPOS=str(y_min),
                                      WIDTH=str(x_max-x_min), HEIGHT=str(y_max-y_min))
            ET.SubElement(text_line, 'String', CONTENT=text)

    return ET.tostring(root, encoding='utf-8').decode('utf-8')


for i in range(1, 51):  # Pages 1 to 50
    filename = f"{i:04d}.png"
    img_path = os.path.join(input_folder, filename)
    print(f"Processing {filename}...")

    # OCR
    result = ocr.ocr(img_path)

    # Get image size
    img = cv2.imread(img_path)
    img_height, img_width = img.shape[:2]

    # Save ALTO XML
    alto_xml = create_alto_xml(result, img_width, img_height)
    xml_path = os.path.join(output_folder, f"{i:04d}.xml")
    with open(xml_path, "w", encoding="utf-8") as f:
        f.write(alto_xml)

    print(f"Saved {xml_path}")

[2025/11/13 12:18:38] ppocr DEBUG: Namespace(help='==SUPPRESS==', use_gpu=False, use_xpu=False, use_npu=False, ir_optim=True, use_tensorrt=False, min_subgraph_size=15, precision='fp32', gpu_mem=500, image_dir=None, page_num=0, det_algorithm='DB', det_model_dir='/Users/joenockels/.paddleocr/whl/det/en/en_PP-OCRv3_det_infer', det_limit_side_len=960, det_limit_type='max', det_box_type='quad', det_db_thresh=0.3, det_db_box_thresh=0.6, det_db_unclip_ratio=1.5, max_batch_size=10, use_dilation=False, det_db_score_mode='fast', det_east_score_thresh=0.8, det_east_cover_thresh=0.1, det_east_nms_thresh=0.2, det_sast_score_thresh=0.5, det_sast_nms_thresh=0.2, det_pse_thresh=0, det_pse_box_thresh=0.85, det_pse_min_area=16, det_pse_scale=1, scales=[8, 16, 32], alpha=1.0, beta=1.0, fourier_degree=5, rec_algorithm='SVTR_LCNet', rec_model_dir='/Users/joenockels/.paddleocr/whl/rec/en/en_PP-OCRv3_rec_infer', rec_image_inverse=True, rec_image_shape='3, 48, 320', rec_batch_num=6, max_text_length=25, rec_ch

**Summary** - This notebook has guided you through importing PaddleOCR, utilising the company's main vision-language English model for text recognition. OpenCV Graphics is used for layout analysis, due to compatibility issues between Mac IOS and paddlex layout detection. This notebook also includes the code necessary for a looped workflow over a sample set of 19th century *Spiritualist Newspaper* images. Future work includes conducting CER tests externally from the model used, as well as conducting eventual comparative findings with the other OCRs tested as part of this notebook series and NLS fellowship.

For any further guidance, comments or questions - contact Dr Joe Nockels at j.nockels@sheffield.ac.uk.